# 06 – Simple Linear Regression

Author: Joe  
Project: AI Foundations – Probability & Statistics

This notebook introduces **simple linear regression**, very much in the style of an
intro stats / TU Dublin MATH4002 module.

We'll cover:

- Scatterplots and correlation
- Simple linear regression model: `Y = a + bX`
- Least squares estimates for slope and intercept
- Fitted values, residuals and residual plots
- Coefficient of determination $R^2$
- Using the model for prediction (with the usual caveats)

The idea is to mirror the R workflow from class, but with NumPy and matplotlib so you can
poke the mechanics more directly.

## Contents
1. [Setup](#1-setup)
2. [Scatterplots & Correlation](#2-scatterplots--correlation)
3. [Simple Linear Regression Model](#3-simple-linear-regression-model)
4. [Least Squares Estimation](#4-least-squares-estimation)
5. [Residuals & Diagnostic Plots](#5-residuals--diagnostic-plots)
6. [Coefficient of Determination $R^2$](#6-coefficient-of-determination-r2)
7. [Prediction & Interpretation](#7-prediction--interpretation)
8. [Practice / TODOs (MATH4002 alignment)](#8-practice--todos-math4002-alignment)


## 1. Setup

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

np.set_printoptions(precision=4, suppress=True)

---
## 2. Scatterplots & Correlation

Regression starts with two quantitative variables measured on the same units (e.g.
hours studied and exam score).

- A **scatterplot** shows the relationship visually.
- The **correlation coefficient** $r$ summarises the strength and direction of the *linear* relationship.

For data points $(x_i, y_i)$, the sample correlation is
\begin{equation}
r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}.
\end{equation}

Here we make a small synthetic dataset to play with.

In [ ]:
# Synthetic example: hours studied vs exam scores
np.random.seed(0)
n = 30
hours = np.random.uniform(0, 10, size=n)
noise = np.random.normal(0, 5, size=n)
scores = 40 + 4 * hours + noise  # roughly linear, plus noise

hours[:5], scores[:5]

In [ ]:
# Scatterplot
plt.scatter(hours, scores)
plt.xlabel('Hours studied')
plt.ylabel('Exam score')
plt.title('Hours vs exam score – scatterplot')
plt.show()

In [ ]:
# Sample correlation
r = np.corrcoef(hours, scores)[0, 1]
r

---
## 3. Simple Linear Regression Model

The simple linear regression model assumes that, for each value of $x$:
\begin{equation}
Y = a + b x + \varepsilon,
\end{equation}
where:

- $a$ is the **intercept** (expected value of $Y$ when $x = 0$),
- $b$ is the **slope** (change in mean $Y$ for a one-unit increase in $x$),
- $\varepsilon$ is a random error term with mean 0.

Given data $(x_i, y_i)$, we estimate $a$ and $b$ using the **least squares** method:
choose $\hat{a}$ and $\hat{b}$ that minimise the sum of squared vertical distances
between the points and the line.

---
## 4. Least Squares Estimation

For sample means $\bar{x}$ and $\bar{y}$, the least squares estimates are
\begin{align}
\hat{b} &= \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}, \\
\hat{a} &= \bar{y} - \hat{b} \bar{x}.
\end{align}

This is what R calculates when you run `lm(y ~ x)`.

In [ ]:
def fit_simple_linear_regression(x: np.ndarray, y: np.ndarray):
    """Return intercept a_hat and slope b_hat for simple linear regression."""
    x = np.asarray(x)
    y = np.asarray(y)
    x_bar = x.mean()
    y_bar = y.mean()
    S_xy = np.sum((x - x_bar) * (y - y_bar))
    S_xx = np.sum((x - x_bar) ** 2)
    b_hat = S_xy / S_xx
    a_hat = y_bar - b_hat * x_bar
    return a_hat, b_hat

a_hat, b_hat = fit_simple_linear_regression(hours, scores)
a_hat, b_hat

In [ ]:
# Plot the fitted line on top of the scatterplot
x_line = np.linspace(hours.min(), hours.max(), 100)
y_line = a_hat + b_hat * x_line

plt.scatter(hours, scores)
plt.plot(x_line, y_line)
plt.xlabel('Hours studied')
plt.ylabel('Exam score')
plt.title('Fitted regression line: score ≈ a + b * hours')
plt.show()

---
## 5. Residuals & Diagnostic Plots

Once we fit the line, we can compute:

- **Fitted values** $\hat{y}_i = \hat{a} + \hat{b} x_i$.
- **Residuals** $e_i = y_i - \hat{y}_i$.

A good regression fit (under the standard assumptions) typically shows:

- No obvious pattern in residuals vs fitted values (no curve or funnel shape).
- Residuals roughly centred around 0.

In MATH4002, you’d often look at these via `plot(lm_model)` in R; here we reproduce
the core ideas with NumPy and matplotlib.

In [ ]:
# Compute fitted values and residuals
y_hat = a_hat + b_hat * hours
residuals = scores - y_hat

list(zip(scores[:5], y_hat[:5], residuals[:5]))

In [ ]:
# Residuals vs fitted values
plt.scatter(y_hat, residuals)
plt.axhline(0)
plt.xlabel('Fitted values (ŷ)')
plt.ylabel('Residuals (e)')
plt.title('Residuals vs fitted values')
plt.show()

In [ ]:
# Histogram of residuals (rough check of normal-ish shape)
plt.hist(residuals, bins=10, density=True)
plt.xlabel('Residual')
plt.ylabel('Density')
plt.title('Residuals histogram')
plt.show()

---
## 6. Coefficient of Determination $R^2$

$R^2$ measures how much of the variability in $Y$ is explained by the linear model.

- Total sum of squares:  
  $$\mathrm{SST} = \sum (y_i - \bar{y})^2.$$
- Residual sum of squares:  
  $$\mathrm{SSE} = \sum (y_i - \hat{y}_i)^2.$$

Then
\begin{equation}
R^2 = 1 - \frac{\mathrm{SSE}}{\mathrm{SST}}.
\end{equation}

Interpretation example: $R^2 = 0.70$ means *about 70% of the variability in $Y$ is explained
by the linear relationship with $X$* (under the model).

In [ ]:
def regression_diagnostics(x: np.ndarray, y: np.ndarray, a_hat: float, b_hat: float):
    x = np.asarray(x)
    y = np.asarray(y)
    y_hat = a_hat + b_hat * x
    residuals = y - y_hat
    y_bar = y.mean()
    SST = np.sum((y - y_bar) ** 2)
    SSE = np.sum(residuals ** 2)
    R2 = 1 - SSE / SST
    return y_hat, residuals, SST, SSE, R2

y_hat, residuals, SST, SSE, R2 = regression_diagnostics(hours, scores, a_hat, b_hat)
SST, SSE, R2

---
## 7. Prediction & Interpretation

Once we have a fitted model
$$\hat{y} = \hat{a} + \hat{b} x,$$
we can use it to **predict** the response for new $x$ values.

Important distinctions (emphasised in many courses):

- A **fitted value** $\hat{y}$ is just a point prediction.
- A full analysis would add a **confidence interval** for the mean response and a
  **prediction interval** for a single new observation (usually done via R in MATH4002).
- Predictions are only sensible within the range of your data (be wary of wild extrapolation).

Here we code a small helper for point predictions and keep the interpretation in words.

In [ ]:
def predict(a_hat: float, b_hat: float, x_new: float) -> float:
    return a_hat + b_hat * x_new

# Example: predicted exam score for 5 hours of study
x_new = 5.0
y_pred = predict(a_hat, b_hat, x_new)
y_pred

Interpretation example (you can adapt this wording for assignments):

> Based on the fitted model, each additional hour of study is associated with an estimated
> increase of $\hat{b}$ points in the exam score, on average. For a student who studies 5 hours,
> the predicted score is roughly $\hat{y}$ points. Actual scores will vary around this line due to
> other factors and random variation.

In a formal solution you'd usually:

- Quote the fitted line (rounded): e.g. `score = 42.3 + 3.8 × hours`.
- Interpret the slope in context.
- Comment on $R^2$ (e.g. "about 65% of variation explained").
- Comment on any obvious patterns in residual plots.

---
## 8. Practice / TODOs (MATH4002 alignment)

This section is where you plug in **real datasets** from your TU Dublin course
(or exam-style questions).

Typical tasks from MATH4002-style material:

- Draw or interpret a scatterplot.
- Compute/interpret the correlation coefficient.
- Fit a simple linear regression line `Y on X`.
- Interpret slope and intercept in context.
- Compute and interpret $R^2$.
- Comment on residual plots (linear pattern? constant spread? outliers?).

### 8.1 Skeleton for a real question

Replace the `...` with actual numbers from a question:

```python
# X: explanatory variable (e.g. hours, temperature)
# Y: response variable (e.g. score, sales)
x = np.array([...], dtype=float)
y = np.array([...], dtype=float)

# 1. Scatterplot
plt.scatter(x, y)
plt.xlabel('X label')
plt.ylabel('Y label')
plt.title('Scatterplot of Y vs X')
plt.show()

# 2. Correlation
r = np.corrcoef(x, y)[0, 1]
print('Correlation r =', r)

# 3. Fit regression line
a_hat, b_hat = fit_simple_linear_regression(x, y)
print('Intercept a_hat =', a_hat)
print('Slope b_hat     =', b_hat)

# 4. Diagnostics
y_hat, residuals, SST, SSE, R2 = regression_diagnostics(x, y, a_hat, b_hat)
print('R^2 =', R2)
```

You can then mirror the R outputs you see in class (`summary(lm(...))`) and use this
notebook to understand each piece numerically and visually.

As you add more concrete examples, this becomes your go-to reference for simple
linear regression, aligned with the way it's taught in your module.